In [1]:
config = "../configs/rec/PP-OCRv5_inf/eslav_PP-OCRv5_mobile_rec.yml"
model_path1 = "../../pretrained/eslav_PP-OCRv5_mobile_rec_pretrained.pdparams"
model_path2 = "../output/eslav_rec_ppocr_v5_exp5/best_accuracy.pdparams"


In [2]:
from merge_utils import build_and_load

d:\Users\baher\anaconda3\envs\paddleocr_gpu\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Skipping import of the encryption module.


In [3]:
model_base = build_and_load(model_path1, config)
model_tuned = build_and_load(model_path2, config)

key is:  backbone.conv1.conv.weight
key is:  backbone.conv1.bn.weight
key is:  backbone.conv1.bn.bias
key is:  backbone.conv1.bn._mean
key is:  backbone.conv1.bn._variance
key is:  backbone.blocks2.0.dw_conv.identity.weight
key is:  backbone.blocks2.0.dw_conv.identity.bias
key is:  backbone.blocks2.0.dw_conv.identity._mean
key is:  backbone.blocks2.0.dw_conv.identity._variance
key is:  backbone.blocks2.0.dw_conv.conv_kxk.0.conv.weight
key is:  backbone.blocks2.0.dw_conv.conv_kxk.0.bn.weight
key is:  backbone.blocks2.0.dw_conv.conv_kxk.0.bn.bias
key is:  backbone.blocks2.0.dw_conv.conv_kxk.0.bn._mean
key is:  backbone.blocks2.0.dw_conv.conv_kxk.0.bn._variance
key is:  backbone.blocks2.0.dw_conv.conv_kxk.1.conv.weight
key is:  backbone.blocks2.0.dw_conv.conv_kxk.1.bn.weight
key is:  backbone.blocks2.0.dw_conv.conv_kxk.1.bn.bias
key is:  backbone.blocks2.0.dw_conv.conv_kxk.1.bn._mean
key is:  backbone.blocks2.0.dw_conv.conv_kxk.1.bn._variance
key is:  backbone.blocks2.0.dw_conv.conv_kxk.2

In [4]:
import os
import random
import paddle.nn as layerlib
import paddle
from paddle.vision import transforms
from PIL import Image

# ----------------------------
# 1. Configuration
# ----------------------------
folder1 = "C:/Users/baher/OneDrive/Desktop/masters/masters_thesis/data/Hand"
folder2 = "C:/Users/baher/OneDrive/Desktop/masters/masters_thesis/master_code/test/cropped_texts"  # <-- Update if needed
num_images_per_folder = 16
batch_size = 1
target_size = (48, 320)

def get_image_paths(folder):
    extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(extensions)]

images1 = get_image_paths(folder1)
images2 = get_image_paths(folder2)

selected1 = random.sample(images1, min(num_images_per_folder, len(images1)))
selected2 = random.sample(images2, min(num_images_per_folder, len(images2)))
all_selected_images = selected1 + selected2
print(f"Total selected images: {len(all_selected_images)}")

transform = transforms.Compose([
    transforms.Resize(target_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ----------------------------
# 2. Hook setup — move outputs to CPU immediately
# ----------------------------
def register_hooks(model, skip_gtc=True):
    outputs = {}
    def make_hook(name):
        def hook(layer, inp, out):
            if name not in outputs:
                outputs[name] = []
            # Move to CPU to avoid GPU memory buildup
            outputs[name].append(out.detach().cpu())
        return hook

    for name, layer in model.named_sublayers():
        if skip_gtc and "gtc" in name:
            continue
        layer.register_forward_post_hook(make_hook(name))
    return outputs

# ----------------------------
# 3. Prepare models
# ----------------------------
def prepare_model(model):
    model.eval()
    use_gtc_attr = hasattr(model.head, 'use_gtc')
    original_use_gtc = getattr(model.head, 'use_gtc', None)
    if use_gtc_attr:
        model.head.use_gtc = False
    return original_use_gtc, use_gtc_attr

orig_gtc_base, has_gtc_base = prepare_model(model_base)
activations_base = register_hooks(model_base)

orig_gtc_tuned, has_gtc_tuned = prepare_model(model_tuned)
activations_tuned = register_hooks(model_tuned)

# ----------------------------
# 4. Forward pass in batches
# ----------------------------
def process_batch(img_paths):
    batch_tensors = []
    for img_path in img_paths:
        img = Image.open(img_path).convert('RGB')
        tensor = transform(img)
        batch_tensors.append(tensor)
    return paddle.stack(batch_tensors)

num_batches = (len(all_selected_images) + batch_size - 1) // batch_size

with paddle.no_grad():
    for i in range(num_batches):
        start = i * batch_size
        end = min(start + batch_size, len(all_selected_images))
        batch_paths = all_selected_images[start:end]
        input_batch = process_batch(batch_paths)
        _ = model_base(input_batch)
        _ = model_tuned(input_batch)
        print(f"Processed batch {i+1}/{num_batches}, shape: {input_batch.shape}")

# ----------------------------
# 5. Restore GTC flags
# ----------------------------
if has_gtc_base:
    model_base.head.use_gtc = orig_gtc_base
if has_gtc_tuned:
    model_tuned.head.use_gtc = orig_gtc_tuned

# ----------------------------
# 6. Concatenate all activations (now on CPU)
# ----------------------------
for name in activations_base:
    activations_base[name] = paddle.concat(activations_base[name], axis=0)
for name in activations_tuned:
    activations_tuned[name] = paddle.concat(activations_tuned[name], axis=0)

print(f"model_base: captured {len(activations_base)} layer outputs")
print(f"model_tuned: captured {len(activations_tuned)} layer outputs")

first_layer_name = next(iter(activations_base.keys()))
print(f"Example output shape from '{first_layer_name}': {activations_base[first_layer_name].shape}")

Total selected images: 128
Processed batch 1/128, shape: [1, 3, 48, 320]
Processed batch 2/128, shape: [1, 3, 48, 320]
Processed batch 3/128, shape: [1, 3, 48, 320]
Processed batch 4/128, shape: [1, 3, 48, 320]
Processed batch 5/128, shape: [1, 3, 48, 320]
Processed batch 6/128, shape: [1, 3, 48, 320]
Processed batch 7/128, shape: [1, 3, 48, 320]
Processed batch 8/128, shape: [1, 3, 48, 320]
Processed batch 9/128, shape: [1, 3, 48, 320]
Processed batch 10/128, shape: [1, 3, 48, 320]
Processed batch 11/128, shape: [1, 3, 48, 320]
Processed batch 12/128, shape: [1, 3, 48, 320]
Processed batch 13/128, shape: [1, 3, 48, 320]
Processed batch 14/128, shape: [1, 3, 48, 320]
Processed batch 15/128, shape: [1, 3, 48, 320]
Processed batch 16/128, shape: [1, 3, 48, 320]
Processed batch 17/128, shape: [1, 3, 48, 320]
Processed batch 18/128, shape: [1, 3, 48, 320]
Processed batch 19/128, shape: [1, 3, 48, 320]
Processed batch 20/128, shape: [1, 3, 48, 320]
Processed batch 21/128, shape: [1, 3, 48, 